# Chapter 2 — Text Classification (Fine-Tuning Basics)

**Goal:** Transform SmolLM from a general-purpose language model into a specialized text classifier.

This is the simplest possible fine-tuning example. By the end you'll understand:
- How fine-tuning changes model behavior (not just knowledge)
- What LoRA is and why we use it
- How to measure if fine-tuning actually worked

**Task:** Classify user messages into 4 categories: `bug`, `feature_request`, `praise`, `question`

## 2.1 Load the Base Model

Same setup as Chapter 1. We start from the raw SmolLM — no prior task training.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import numpy as np

BASE_MODEL = "HuggingFaceTB/SmolLM-135M"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
if device == "cpu":
    model = model.to(device)

print(f"Model loaded on {device}")

## 2.2 Create Training Data

We generate a small synthetic dataset. Each example is a **prompt → completion** pair. The model will learn to complete the prompt with the correct label.

In a real project you'd label real data. Here we use templates so the patterns are clean and the model can learn them reliably.

In [ ]:
def make_example(text, label):
    prompt = f"Classify: {text}\nLabel:"
    completion = f" {label}"
    return {"prompt": prompt, "completion": completion}

raw_data = []

# --- BUG: something is broken / not working as expected ---
bug_seeds = [
    "app crashes when I tap", "screen freezes after", "notifications not showing up on",
    "upload fails with error", "dark mode doesn't apply to", "keyboard covers the input",
    "video playback stops randomly after", "can't scroll past", "logs me out every time",
    "shows blank white page on", "permission denied even though enabled on",
    "search results not updating when", "app drains battery in background since",
    "language preference doesn't save after", "images not loading on",
    "two-factor code never arrives on", "calendar keeps duplicating events on",
    "mic stops working mid-call on", "export produces corrupted file on",
    "filter returns wrong results on", "drag and drop broken on",
    "password reset link always expired on", "push notifications arrive hours late on",
    "screen goes black after splash on", "draft wipes all content when",
]
for seed in bug_seeds:
    raw_data.append(make_example(f"{seed} the profile page", "bug"))
    raw_data.append(make_example(f"{seed} latest update", "bug"))
    raw_data.append(make_example(f"{seed} my Android phone", "bug"))
    raw_data.append(make_example(f"{seed} the checkout screen", "bug"))

# --- FEATURE REQUEST: asking for something new that doesn't exist yet ---
feature_seeds = [
    "please add dark mode for", "would love a bulk delete option in",
    "can we get offline mode for", "please add support for multiple",
    "widgets on home screen for", "need a way to export data as",
    "voice commands would improve", "please integrate with Google",
    "add an option to pin", "would like to customize notifications for",
    "add biometric lock for", "keyboard shortcuts needed in",
    "please add schedule-send for", "support right-to-left text in",
    "add a reading mode in", "wish I could merge duplicate",
    "add markdown support in", "please add recurring tasks to",
    "would like to share lists with", "add a built-in QR scanner in",
    "please support split-screen on", "need an undo-send feature for",
    "add playback speed control for", "please add currency converter in",
    "wish I could set custom swipe gestures for",
]
for seed in feature_seeds:
    raw_data.append(make_example(f"{seed} the app", "feature_request"))
    raw_data.append(make_example(f"{seed} my account", "feature_request"))
    raw_data.append(make_example(f"{seed} the dashboard", "feature_request"))
    raw_data.append(make_example(f"{seed} the editor", "feature_request"))

# --- PRAISE: compliment, positive experience ---
praise_seeds = [
    "this app is amazing,", "love the new update,", "best productivity tool",
    "the design is beautiful and", "customer support was incredibly helpful with",
    "switched from a competitor and", "speed improvements are really noticeable in",
    "recommended this to my whole team because", "you nailed the onboarding for",
    "five stars, does exactly what I need for", "tutorial was clear and I got started with",
    "love how lightweight the app is,", "sync between devices works flawlessly for",
    "search feature finds everything instantly in", "thank you for listening to feedback about",
    "been using this daily for years and", "widgets are gorgeous and functional on",
    "privacy settings are exactly what I wanted for", "great value, honestly underpriced for",
    "the new themes are fantastic, especially", "collaboration features work perfectly for",
    "works offline without any issues on", "best-in-class accessibility features in",
]
for seed in praise_seeds:
    raw_data.append(make_example(f"{seed} so happy with it", "praise"))
    raw_data.append(make_example(f"{seed} great job team", "praise"))
    raw_data.append(make_example(f"{seed} truly impressed", "praise"))
    raw_data.append(make_example(f"{seed} keep up the good work", "praise"))

# --- QUESTION: asking how/where/when/what — NOT reporting a problem or requesting a feature ---
question_seeds = [
    "how do I change my account", "where can I find the billing",
    "what permissions does the app need for", "does the free tier include",
    "how many devices can I use with one", "is my data encrypted on",
    "how do I invite team members to a", "what happens to my data if I cancel",
    "can I use this app for commercial", "how long does export take for",
    "is there a storage limit per", "how do I set up two-factor auth on",
    "does the API have rate limits for", "can I import data from the old",
    "what file formats are supported for", "how do I contact support about",
    "is there a student discount for", "which regions are your servers in for",
    "do you comply with GDPR in", "how often is the app updated with",
    "can I customize the dashboard in", "is there a way to try premium features before",
    "what is the refund policy for", "how do I transfer workspace ownership for",
]
for seed in question_seeds:
    raw_data.append(make_example(f"{seed} email address", "question"))
    raw_data.append(make_example(f"{seed} my profile", "question"))
    raw_data.append(make_example(f"{seed} the mobile app", "question"))
    raw_data.append(make_example(f"{seed} my subscription", "question"))

# Shuffle and split
np.random.seed(42)
np.random.shuffle(raw_data)
split = int(0.8 * len(raw_data))
train_data = raw_data[:split]
eval_data = raw_data[split:]

print(f"Train: {len(train_data)} ({len(train_data)//4} per class)")
print(f"Eval:  {len(eval_data)} ({len(eval_data)//4} per class)")
print(f"\nSample: {train_data[0]['prompt']} → {train_data[0]['completion']}")

## 2.3 Tokenize the Data

Convert text to token IDs. We set prompt tokens to `-100` in the labels so the loss is **only** computed on the label portion — the model learns to produce the correct classification, not to regurgitate the input.

In [ ]:
def tokenize(example):
    """Tokenize prompt and completion separately, append EOS so the model
    learns to stop after the label instead of rambling."""
    prompt_tokens = tokenizer(
        example["prompt"], truncation=True, max_length=128, padding=False,
    )
    completion_tokens = tokenizer(
        example["completion"], truncation=True, max_length=32, padding=False,
    )

    input_ids = (
        prompt_tokens["input_ids"]
        + completion_tokens["input_ids"]
        + [tokenizer.eos_token_id]
    )
    attention_mask = [1] * len(input_ids)
    # Mask prompt tokens — loss only on completion + EOS
    labels = (
        [-100] * len(prompt_tokens["input_ids"])
        + completion_tokens["input_ids"]
        + [tokenizer.eos_token_id]
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

train_dataset = Dataset.from_list(train_data).map(tokenize, remove_columns=["prompt", "completion"])
eval_dataset = Dataset.from_list(eval_data).map(tokenize, remove_columns=["prompt", "completion"])

print(f"Tokenized {len(train_dataset)} train / {len(eval_dataset)} eval examples")
sample = train_dataset[0]
masked = sum(1 for l in sample["labels"] if l == -100)
print(f"Sample: {len(sample['input_ids'])} tokens, {masked} masked, {len(sample['input_ids']) - masked} label tokens (incl. EOS)")

## 2.4 Apply LoRA

**LoRA** (Low-Rank Adaptation) freezes the original model weights and adds small trainable "adapters" to specific layers. Instead of updating all 135M parameters, we train only ~400K new parameters — that's 0.3% of the model.

This means:
- Training is 3-5x faster
- Memory usage drops dramatically (fits on 4GB GPU)
- The adapter file is ~2MB instead of ~270MB
- You can swap adapters for different tasks without reloading the base model

In [12]:
lora_config = LoraConfig(
    r=8,                # rank — higher = more capacity, lower = more compression
    lora_alpha=16,      # scaling factor
    lora_dropout=0.05,  # regularization
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / Total: {total:,} ({100*trainable/total:.1f}%)")

Trainable: 460,800 / Total: 134,975,808 (0.3%)


## 2.5 Train

We train for just a few epochs. On a free Colab GPU this takes ~2-3 minutes.

In [ ]:
def collate_fn(batch):
    """Pad a batch to the longest sequence. Labels padded with -100 (ignore index)."""
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append(x["attention_mask"] + [0] * pad_len)
        labels.append(x["labels"] + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_mask),
        "labels": torch.tensor(labels),
    }

training_args = TrainingArguments(
    output_dir="./chapters/02-classification/checkpoint",
    num_train_epochs=12,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    learning_rate=1e-4,
    weight_decay=0.01,
    fp16=(device == "cuda"),
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
)

print("Starting training...")
trainer.train()

## 2.6 Evaluate — Before vs After

This is the most important section. We compare the **same prompts** on the **same model** before and after fine-tuning. The difference tells us whether training worked.

In [ ]:
test_texts = [
    "the app keeps crashing when I try to upload a photo",
    "would be great if we could have dark mode",
    "this is honestly the best note-taking app I've used",
    "how do I reset my password from the mobile app",
    "getting a blank screen after the latest update",
    "please add markdown support to the editor",
    "the onboarding was so smooth, loved every bit of it",
    "does the free version include cloud sync",
]

expected = ["bug", "feature_request", "praise", "question",
            "bug", "feature_request", "praise", "question"]

VALID_LABELS = {"bug", "feature_request", "praise", "question"}

def classify(text):
    prompt = f"Classify: {text}\nLabel:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=3,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,  # stop at EOS
        )
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract and match against valid labels
    if "Label:" in full:
        raw = full.split("Label:")[-1].strip()
        for label in VALID_LABELS:
            if raw.startswith(label):
                return label
    return raw.split()[0] if raw and raw.split() else "???"

print("Predictions after fine-tuning:\n")
correct = 0
for text, exp in zip(test_texts, expected):
    pred = classify(text)
    ok = "OK" if pred == exp else f"WRONG (expected {exp})"
    if pred == exp:
        correct += 1
    print(f"  [{ok}] {text[:60]}...")
    print(f"         → {pred}\n")

print(f"Accuracy: {correct}/{len(test_texts)} ({100*correct/len(test_texts):.0f}%)")

## 2.7 Save the Adapter

Only the LoRA weights need to be saved — the base model stays unchanged. This tiny file (~2MB) is all you need to reconstruct the fine-tuned model later.

In [ ]:
adapter_path = "./adapter"
model.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

## 2.8 What Just Happened?

| Before fine-tuning | After fine-tuning |
|---|---|
| Model outputs random plausible text | Model outputs one of 4 labels |
| No concept of classification | Learned the `Classify: ... Label:` pattern |
| ~135M params, general purpose | Same base + ~400K LoRA params, task-specific |

**Key insight:** We didn't teach the model *facts*. We taught it a *behavior* — given a specific prompt format, respond with a specific output format. This is what fine-tuning is best at.

In Chapter 3 we'll extend this to a harder task: extracting structured JSON from free text.